In [1]:
from diffusers import UNet2DConditionModel

扩散模型在最初使用Unet的原因可能是Unet模型的输入和输出图片大小是一样的，Unet是扩散系统的重要组成部分，因为其促进了实际的扩散过程。在Diffusers中Unet模型有几种变体，具体取决于它的维数以及是否是条件模型，下面是一个2D Unet条件模型。

In [37]:
unet=UNet2DConditionModel(sample_size=64,cross_attention_dim=768).to("cuda:0")

sample_size=64，这是由于VAE将[bs,3,512,512]的噪声编码为[bs,4,64,64]，也正因此
in_channels=4
out_channels=4

down_block_types,mid_block_typeup_block_types直接定义Unet的结构

down_block_types: Tuple[str] = (
            "CrossAttnDownBlock2D",
            "CrossAttnDownBlock2D",
            "CrossAttnDownBlock2D",
            "DownBlock2D",
        )
结构中的CrossAttnDownBlock2D是旧版Diffusers中的SpatialTransformer，
前三个CrossAttnDownBlock2D会发生downsample，第四个DownBlock并不发生downsample，Unet下采样过程会发生三次downsample


想一想，Unet的特点是输入输出尺寸一致，那么有三次下采样一定会有三次上采样

up_block_types: Tuple[str] = (
            "UpBlock2D", 
            "CrossAttnUpBlock2D", 
            "CrossAttnUpBlock2D", 
            "CrossAttnUpBlock2D"),

在这里UpBlock2D和前两个CrossAttnUpBlock2D会发生UpSample第三个CrossAttnUpBlock2D不发生UpSample

only_cross_attention:一定是False，文本控制必然会发生cross_attention(待确认)

block_out_channels:这里先理解为down_block_types每一层的输出通道数 (320, 640, 1280, 1280)

layers_per_block:CrossAttnDownBlock2D中包含多少对ResnetBlock+Transformer2DModel

attention_head_dim和cross_attention_dim 常见的头数就是8，cross_attention_dim表示文本编码的维度

下面要看一看forward函数
主要是三个参数
sample:latent
timestep:扩散去噪从第几步开始
encoder_hidden_states：控制向量，文本编码

基本使用

In [ ]:
import torch

batch_size,in_channels,sample_size=2,4,64

latents=torch.randn(size=[batch_size,in_channels,sample_size,sample_size],dtype=torch.float32)

time_steps=torch.randint(0,1000,size=[batch_size])

sequence_length=20

encoder_hidden_states=torch.randn(size=[batch_size,sequence_length,768],dtype=torch.float32)

out_put=unet(sample=latents,timestep=time_steps,encoder_hidden_states=encoder_hidden_states)

print("ok")
print(out_put.sample.shape)

现在已经大致明确了Unet2DConditionModel的组成，知道了输入输出大致的形状，
接下来继续深入剖析一下Unet2DConditionModel内部细节

我们做如下定义：
设Unet2DConditionModel为一级结构，其内部包含down_blocks,middle_block以及up_blocks，它们各自为一个nn.ModuleList

每一个ModuleList为二级结构，其内部包含CrossAttentionBlock2D,DownBlock2D等

每一个CrossAttentionBlock(等)为一个三级结构，其中包含ResNetBlock2D，Transformer2DModel，DownSample2D(注意这里要和DownBlock2D区分开来)等

每一个四级结构中又包含若干五级结构，如ResNetBlock2D中包含若干残差块，Transformer2DModel中包含BasicTransformer(其实就是常见的Transformer块)



以CrossAttentionBlock2D为例，其class，init函数的主要功能就是组合ResNetBlock2D、Transformer2D以及DownSample2D块

下面，我们更想做的事是看一看BasicTransformer中的一些细节

BasicTransformerBlock还是先做归一化，在Transformer中最重要的其实是Attention是怎么操作的:

我们实例化一个Attention层

hidden_states(q):[batch_size,in_channels,height,width]

encoder_hidden_states(k/v):[batch_size,squence_length,feature_dim]

在使用的时候要求：
Attention(query_dim=in_channels,cross_attention_dim=feature_dim)

再来回顾一下Attention是怎么工作的：
首先拿到hidden_states对其形状做变化，变为[batch_size,in_channels,height*width]然后再调整相关维度等

接下来将hidden_states,encoder_hidden_states投影到相同的feature(cross_attention_dim->inner_dim)，以便做后续的点积操作等

在完成注意力机制加权运算后，还要做一次投影以及形状转化操作(目的就是将输出形状再次还原为[batch_size,in_channels,height,width])

最后输出，这就是Attention操作不改变hidden_states大小的原理

最后要说明的是，qky做完注意力机制的操作后形状保持和原来的q一致




In [ ]:

from diffusers.models.attention import Attention

# sameple_size=64
image_embedding=torch.randn(size=[batch_size,in_channels,sample_size,sample_size],dtype=torch.float32)

cond_embdedding=torch.randn(size=[batch_size,sequence_length,768],dtype=torch.float32)

atten=Attention(query_dim=4,cross_attention_dim=768)

atten_out=atten(hidden_states=image_embedding,encoder_hidden_states=cond_embdedding)#encoder_hidden_states=cond_embdedding)

print(atten_out.shape)

BasicTransformerBlock的整体使用

In [ ]:
from diffusers.models.attention import BasicTransformerBlock

# 在初始化的时候只需要确定dim: int,num_attention_heads，attention_head_dim 三个参数，其他数字都可以用默认值

# dim现在研究了一下是h*w
transformer_block=BasicTransformerBlock(dim=64*64,num_attention_heads=10,attention_head_dim=768//8)

# num_attention_heads和attention_head_dim控制的只是cross_attention_dim,并没有改变inner_dim

print(transformer_block)

In [ ]:
tansformer_output=transformer_block(hidden_states=image_embedding.reshape(image_embedding.shape[0],image_embedding.shape[1],-1),encoder_hidden_states=cond_embdedding)
print(tansformer_output.shape)

看一下怎么把每一层输出的结果给拿出来(下面是我自己想的)

In [52]:
import torch
from torch import nn

batch_size,in_channels,sample_size=2,4,64

x=torch.randn(size=[batch_size,in_channels,sample_size,sample_size],dtype=torch.float32,device="cuda:0")

time_steps=torch.randint(0,1000,size=[batch_size],device="cuda:0")

sequence_length=20

y=torch.randn(size=[batch_size,sequence_length,768],dtype=torch.float32,device="cuda:0")


(conv_in): Conv2d(4, 320, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (time_proj): Timesteps()
    (time_embedding): TimestepEmbedding(

In [75]:
out_1=unet.conv_in(x)
print("out_1.shape:",out_1.shape)

time_steps_1=unet.time_proj(time_steps)

print(time_steps_1.shape)

timestep_embedding=unet.time_embedding(time_steps_1)

print(timestep_embedding.shape)

out_x=[out_1]
for i,block in enumerate(unet.down_blocks):
    out_x=block(out_x[0],timestep_embedding,y)

    print(out_x[0].shape)

# out=unet.down_blocks[0](out_x,timestep_embedding,y)



out_1.shape: torch.Size([2, 320, 64, 64])
torch.Size([2, 320])
torch.Size([2, 1280])
torch.Size([2, 320, 32, 32])
torch.Size([2, 640, 16, 16])
torch.Size([2, 1280, 8, 8])
torch.Size([2, 1280, 8, 8])


C:\Users\Shipu\anaconda3\envs\diffusers_use\lib\site-packages\diffusers\models\unets\unet_2d_blocks.py:1360: FutureWarning: `scale` is deprecated and will be removed in version 1.0.0. The `scale` argument is deprecated and will be ignored. Please remove it, as passing it will raise an error in the future. `scale` should directly be passed while calling the underlying pipeline component i.e., via `cross_attention_kwargs`.
  deprecate("scale", "1.0.0", deprecation_message)


In [78]:
from torch import nn
attention=nn.MultiheadAttention

C:\Users\Shipu\anaconda3\envs\diffusers_use\lib\site-packages\torch\nn\modules\transformer.py:306: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


Transformer(
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=512, bias=True)
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
    (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (decoder): TransformerDecoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerDecoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, o

下一篇参考:Unet3DConditionModel